In [1]:
import pandas as pd

service = pd.read_csv("data/water/servicio_agua_red_publica_cngmd2023_csv/conjunto_de_datos/servagua_cngmd2023.csv")
water_supplied = pd.read_csv("data/water/servicio_agua_red_publica_cngmd2023_csv/conjunto_de_datos/admnlocs_cngmd2023.csv")
water_info = pd.read_csv("data/water/servicio_agua_red_publica_cngmd2023_csv/conjunto_de_datos/admnapsa_cngmd2023.csv")

# WATER SERVICE HERE

## Var names
- mnpio: municipality -- Municipality ID code
- ag_servi: water_service -- Whether the municipality has public water network service (binary)
- pobl_prc: population_pct_covered -- Percentage of population with access to public water network
- no_sabe: unknown	-- Whether coverage percentage is unknown

In [2]:
service.head()

,mnpio,ag_servi,pobl_prc,no_sabe
0,1001,1,99.00,0
1,1002,1,75.00,0
2,1003,1,97.85,0
3,1004,1,98.00,0
4,1005,1,98.00,0


In [3]:
service = service.rename(columns={'mnpio': 'CVEGEO'})


In [6]:
print(service.head())
print('-------')
print(service['CVEGEO'].nunique)

   CVEGEO  ag_servi  pobl_prc  no_sabe
0    1001         1     99.00        0
1    1002         1     75.00        0
2    1003         1     97.85        0
3    1004         1     98.00        0
4    1005         1     98.00        0
-------
<bound method IndexOpsMixin.nunique of 0        1001
1        1002
2        1003
3        1004
4        1005
        ...  
2470    32054
2471    32055
2472    32056
2473    32057
2474    32058
Name: CVEGEO, Length: 2475, dtype: int64>


In [11]:
key = pd.read_csv("data/covariates/ethan_final_key_df_head.csv")

print(key.columns.tolist())


key["CVEGEO"] = key["CVEGEO"].astype(str).str.zfill(5)
service["CVEGEO"] = service["CVEGEO"].astype(str).str.zfill(5)


service = key[["CVEGEO"]].merge(
    service,
    on="CVEGEO",
    how="left"
)

print(service.head())
print(service["CVEGEO"].nunique())

print(service.columns.tolist())

['CVEGEO', 'NOMGEO', 'homicidio_doloso_2025', 'robo_veh_violencia_2025', 'n_inseguro_popw']
  CVEGEO  ag_servi  pobl_prc  no_sabe
0  01005       1.0      98.0      0.0
1  01008       1.0      98.0      0.0
2  01001       1.0      99.0      0.0
3  01004       1.0      98.0      0.0
4  01011       1.0       0.0      1.0
2478
['CVEGEO', 'ag_servi', 'pobl_prc', 'no_sabe']


In [12]:
covariate_cols = ['ag_servi', 'pobl_prc', 'no_sabe']

for col in covariate_cols:
    service[f"no_{col}"] = service[col].isna().astype(int)
    service[col] = service[col].fillna(0)


ordered_cols = ["CVEGEO"]
for col in covariate_cols:
    ordered_cols += [col, f"no_{col}"]

service = service[ordered_cols]


In [18]:
print(service)

     CVEGEO  ag_servi  no_ag_servi  pobl_prc  no_pobl_prc  no_sabe  no_no_sabe
0     01005       1.0            0      98.0            0      0.0           0
1     01008       1.0            0      98.0            0      0.0           0
2     01001       1.0            0      99.0            0      0.0           0
3     01004       1.0            0      98.0            0      0.0           0
4     01011       1.0            0       0.0            0      1.0           0
...     ...       ...          ...       ...          ...      ...         ...
2473  32007       1.0            0      98.0            0      0.0           0
2474  32047       1.0            0      97.0            0      0.0           0
2475  32012       1.0            0      80.0            0      0.0           0
2476  32006       1.0            0      90.0            0      0.0           0
2477  32011       1.0            0      90.0            0      0.0           0

[2478 rows x 7 columns]


In [19]:
print(service['no_ag_servi'].nunique())
print(service['no_pobl_prc'].nunique())
print(service['no_no_sabe'].nunique())

2
2
2


In [20]:
service.to_csv("data/covariates/water-service.csv", index=False)